### Initial data reading and analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
from sklearn import linear_model
import lightgbm as lgbm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [2]:
from pathlib import Path
print(Path.cwd())
DATA_DIR = Path("../data")

/Users/danylodubas/Documents/Coding/qrt_challenge_data_2026/notebooks


In [3]:
X_train = pd.read_csv(DATA_DIR / 'X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv(DATA_DIR / 'X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv(DATA_DIR / 'y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv',index_col='ROW_ID')

In [4]:
X_train.columns
X_train["ALLOCATION"].describe()
X_train.head()

,TS,ALLOCATION,RET_20,RET_19,RET_18,RET_17,RET_16,RET_15,RET_14,RET_13,...,SIGNED_VOLUME_8,SIGNED_VOLUME_7,SIGNED_VOLUME_6,SIGNED_VOLUME_5,SIGNED_VOLUME_4,SIGNED_VOLUME_3,SIGNED_VOLUME_2,SIGNED_VOLUME_1,MEDIAN_DAILY_TURNOVER,GROUP
ROW_ID,,,,,,,,,,,,,,,,,,,,,
0,DATE_0001,ALLOCATION_01,-0.018192,-0.000306,-0.006881,-0.002393,0.000507,-0.001270,-0.002539,0.002830,...,0.818730,0.941014,0.714129,-0.323847,0.525097,0.363601,-0.219328,NaN,0.096905,1
1,DATE_0001,ALLOCATION_02,-0.006394,-0.001059,0.001565,0.000033,0.002829,0.001725,0.000875,-0.002160,...,-1.390336,-0.651784,-0.896826,-0.636931,-1.074450,-0.748884,-0.718912,NaN,0.009974,4
2,DATE_0001,ALLOCATION_03,-0.016587,-0.004517,-0.005306,0.004314,0.006471,-0.005868,-0.005030,-0.001488,...,0.961318,0.452482,1.588321,0.790039,1.394445,0.493521,0.268094,NaN,0.044186,1
3,DATE_0001,ALLOCATION_04,-0.005344,0.002790,0.006937,-0.004246,-0.005051,-0.000330,-0.000117,-0.005209,...,-0.483377,-0.565114,-0.631710,-0.663300,-1.615905,-0.959046,-0.478789,NaN,0.001150,2
4,DATE_0001,ALLOCATION_05,-0.010506,-0.005491,0.007752,-0.012299,0.002191,0.003282,0.000495,-0.003489,...,0.268005,0.757707,1.524626,1.565541,1.563963,1.063209,0.921333,NaN,NaN,4


In [24]:
pd.set_option("display.max_rows", None)

counts = (
    X_train.groupby(["TS", "GROUP"])
           .size()
           .reset_index(name="count")
           .sort_values("TS")
)

ts_counts = counts.groupby("TS").size()
ts_groups = (
    X_train.groupby("TS")["GROUP"]
         .nunique()
         .value_counts()
         .sort_index()
)

print(ts_groups)

print(X_train.groupby(["ALLOCATION"]).size().sort_values())

GROUP
1     522
2     130
3     597
4    1273
Name: count, dtype: int64
ALLOCATION
ALLOCATION_14       19
ALLOCATION_46       19
ALLOCATION_244    1431
ALLOCATION_241    1431
ALLOCATION_234    1431
ALLOCATION_230    1431
ALLOCATION_23     1431
ALLOCATION_228    1431
ALLOCATION_216    1431
ALLOCATION_215    1431
ALLOCATION_214    1431
ALLOCATION_205    1431
ALLOCATION_202    1431
ALLOCATION_20     1431
ALLOCATION_01     1431
ALLOCATION_188    1431
ALLOCATION_178    1431
ALLOCATION_177    1431
ALLOCATION_175    1431
ALLOCATION_173    1431
ALLOCATION_170    1431
ALLOCATION_167    1431
ALLOCATION_166    1431
ALLOCATION_165    1431
ALLOCATION_163    1431
ALLOCATION_161    1431
ALLOCATION_25     1431
ALLOCATION_195    1431
ALLOCATION_251    1431
ALLOCATION_254    1431
ALLOCATION_94     1431
ALLOCATION_92     1431
ALLOCATION_91     1431
ALLOCATION_82     1431
ALLOCATION_80     1431
ALLOCATION_72     1431
ALLOCATION_71     1431
ALLOCATION_67     1431
ALLOCATION_66     1431
ALLOCATION_55     14

In [8]:
print(counts)

             TS  GROUP  count
0     DATE_0001      1     70
1     DATE_0001      2     70
2     DATE_0001      3     70
3     DATE_0001      4     66
4     DATE_0002      1     70
5     DATE_0002      2     70
6     DATE_0002      3     70
7     DATE_0002      4     66
8     DATE_0003      3     69
9     DATE_0004      2     69
10    DATE_0004      3     69
11    DATE_0004      4     66
12    DATE_0005      1     70
13    DATE_0005      2     70
14    DATE_0005      3     70
15    DATE_0005      4     66
16    DATE_0006      3     65
17    DATE_0007      1     70
18    DATE_0007      2     70
19    DATE_0007      3     70
20    DATE_0007      4     66
21    DATE_0008      3     69
22    DATE_0009      2     70
23    DATE_0009      3     70
24    DATE_0009      4     66
25    DATE_0010      1     70
26    DATE_0010      2     70
27    DATE_0010      3     70
28    DATE_0010      4     66
29    DATE_0011      3     69
30    DATE_0012      2     70
31    DATE_0012      3     70
32    DATE